In [1]:
# Setting up the file path
import sys
import os
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.append(parent_dir)

In [2]:
import comet_ml
from matplotlib import pyplot as plt
from pytorch_lightning.loggers import CometLogger
from copy import deepcopy
import numpy as np
import gymnasium as gym
from itertools import product
from RL4CRN.Input_Output_Rxn_Networks.IOCRN_MassAction import IOCRN_MassAction
from RL4CRN.Environments.CRNEnvironment import CRNEnvironment
from RL4CRN.Environments.VecCRNEnvironment import VecCRNEnvironment
from RL4CRN.Agents.RandomAgent import RandomAgent
from RL4CRN.Rewards.Transients import dynamic_tracking_error

In [3]:
# Set the logger to use Comet
api_key = "o77J6VCMDamustkfJuMXZ2jdV"
logger = CometLogger(
    api_key=api_key,
    project="Testing_Rendering",        
    workspace="maurice-filo" 
)
logger = logger.experiment

COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.
COMET INFO: Experiment is live on comet.com https://www.comet.com/maurice-filo/testing-rendering/2b2b065b37124d65a9136308539547dc



In [4]:
# Construct a basic CRN
species_labels = ['X_1', 'X_2', 'Z_1', 'Z_2']
inputs_labels = ['u_1', 'u_2', 'u_3']
stoichiometry_reactants = np.array([[0], [0], [1], [0]], dtype=np.int8)
stoichiometry_products = np.array([[1], [0], [1], [0]], dtype=np.int8)
parameters = np.array([1], dtype=np.float32)
input_influence_matrix = np.array([[0], [0], [0]], dtype=np.int8)
outputs = np.array([2], dtype=np.int8)
IOCRN_AIF = IOCRN_MassAction(stoichiometry_reactants, stoichiometry_products, parameters, input_influence_matrix, outputs, species_labels, inputs_labels)

In [5]:
# Add reactions
k_1 = 1; gamma_1 = 1; gamma_2 = 0.1; mu = 10; theta = 1; eta = 0.1
IOCRN_AIF.add_reaction({'reactants index': 1, 'products index': 6, 'input influence index': 0, 'rate constant': k_1})
IOCRN_AIF.add_reaction({'reactants index': 1, 'products index': 0, 'input influence index': 1, 'rate constant': gamma_1})
IOCRN_AIF.add_reaction({'reactants index': 2, 'products index': 0, 'input influence index': 2, 'rate constant': gamma_2})
IOCRN_AIF.add_reaction({'reactants index': 0, 'products index': 3, 'input influence index': 3, 'rate constant': mu})
IOCRN_AIF.add_reaction({'reactants index': 2, 'products index': 11, 'input influence index': 0, 'rate constant': theta})
IOCRN_AIF.add_reaction({'reactants index': 13, 'products index': 0, 'input influence index': 0, 'rate constant': eta})
print('Initial CRN:')
IOCRN_AIF.print_reactions()

Initial CRN:
Inputs: ['u_1', 'u_2', 'u_3'] 
Species: ['X_1', 'X_2', 'Z_1', 'Z_2'] 
Output Species: ['X_2'] 
Reaction 0: Z_1 -> X_1 + Z_1 ; Rate Constant: 1.0 
Reaction 1: X_1 -> X_1 + X_2 ; Rate Constant: 1.0 
Reaction 2: X_1 -> 0 ; Rate Constant: 1.0u_1 
Reaction 3: X_2 -> 0 ; Rate Constant: 0.1u_2 
Reaction 4: 0 -> Z_1 ; Rate Constant: 10.0u_3 
Reaction 5: X_2 -> X_2 + Z_2 ; Rate Constant: 1.0 
Reaction 6: Z_1 + Z_2 -> 0 ; Rate Constant: 0.1 



In [6]:
# Create an environment
max_num_reactions = 2
n_samples = 100
N_CPUs = 128
CRN_template = deepcopy(IOCRN_AIF)
vec_env = VecCRNEnvironment([CRNEnvironment(CRN_template, max_num_reactions, logger=logger, logger_schedule=1) for _ in range(n_samples)], N_CPUs=N_CPUs, logger=logger)

In [ ]:
# Create a random agent
random_agent = RandomAgent(vec_env.envs[0], max_rate_constant=10, logger=logger)

In [8]:
# Create the reward function
def compute_reward(state):
    nums = [0.5, 1, 1.5]
    u = np.array(list(product(nums, repeat=len(nums))), dtype=np.float32)
    initial_condition = np.array([0, 0, 0, 0], dtype=np.float32)
    time_horizon = np.linspace(0, 150, 1000, dtype=np.float32)
    r = u[:,2] * state.parameters[4]
    return dynamic_tracking_error(state, u, initial_condition, time_horizon, r, threshold=1000)

In [9]:
# Run the forward pass
epoch_num = 50
render_schedule = 10
for i in range(epoch_num):
    vec_env.reset()
    for j in range(max_num_reactions):
        actions = [random_agent.act() for _ in range(n_samples)]
        out = vec_env.step(actions)
    rewards = vec_env.get_reward(compute_reward)
    if i % render_schedule == 0:
        vec_env.render(rewards, mode='logger_image')

In [10]:
# Render the environment in human mode
vec_env.envs[0].render(mode='human')

Inputs: ['u_1', 'u_2', 'u_3'] 
Species: ['X_1', 'X_2', 'Z_1', 'Z_2'] 
Output Species: ['X_2'] 
Reaction 0: Z_1 -> X_1 + Z_1 ; Rate Constant: 1.0 
Reaction 1: X_1 -> X_1 + X_2 ; Rate Constant: 1.0 
Reaction 2: X_1 -> 0 ; Rate Constant: 1.0u_1 
Reaction 3: X_2 -> 0 ; Rate Constant: 0.1u_2 
Reaction 4: 0 -> Z_1 ; Rate Constant: 10.0u_3 
Reaction 5: X_2 -> X_2 + Z_2 ; Rate Constant: 1.0 
Reaction 6: Z_1 + Z_2 -> 0 ; Rate Constant: 0.1 
Reaction 7: 2 Z_1 -> X_2 ; Rate Constant: 5.008726783147645 
Reaction 8: X_1 -> Z_1 ; Rate Constant: 5.585799843020536 

